# 4 · Energetic data analysis

This notebook turns the **62,800 configurations per host–guest pair** generated
in the previous stage into a single **reported minimum binding energy** per pair.
It implements methodology **§6**, documented in
[`energetic-data-analysis.md`](../energetic-data-analysis/energetic-data-analysis.md).

The pipeline is:

1. **MD snapshot extraction** — pull the guest plus its host unit cell out of each
   retained LAMMPS frame and write AMS single-point (SPE) inputs.
2. **SPE screening** — read the GFN1-xTB total energy of every configuration,
   apply the steric-clash filter, and select the lowest-energy candidates.
3. **Geometry optimization** — relax the selected candidates.
4. **Final binding energy** — `BE = E_complex − (E_host + E_guest)`; the reported
   value is the minimum over all relaxed candidates.

> **⚠ Withheld bulk data.** The raw SPE / geo-opt `.out` files (millions of them)
> are far too large to commit and are *not* included here. The cells that *parse*
> those outputs therefore cannot be re-run from this repository alone; they are
> kept, documented, and runnable against your own calculation outputs. The
> committed key results (final binding energies, rankings, correlations) are
> consumed in [`selectivity-analysis`](./selectivity-analysis.ipynb).

Reusable, non-thesis-specific file helpers come from
[`mof-guest-toolkit`](https://github.com/adricu12/mof-guest-toolkit); the
thesis-specific binding-energy and selectivity functions are in
[`pipeline_utils.py`](./pipeline_utils.py).

---
## 5.2.3 MD trajectory snapshot extraction

## Single-point inputs for the configuration pool

Every sampled configuration — from both SRD and MD — gets a GFN1-xTB single-point
energy (SPE); the inputs are built one pool at a time. The cells below read each
**SRD** HDF5 file from `051_SRD/` and write one AMS single-point `.run` (plus a
CIF) per configuration into `061_SPE_COMPLEXES/`.

> **⚠ Requires withheld data.** The SRD HDF5 pools are not committed (one small
> sample is included; see [`thesis/README.md`](../thesis/README.md)).

In [ ]:
# ! pip install h5py
from pathlib import Path
import h5py
import numpy as np
import re

from mof_toolkit import gen_inp_file, frac_to_cart

In [ ]:
# -----------------------------
# Settings
# -----------------------------

h5_root = Path("../thesis/051_SRD").resolve()
out_root = Path("../thesis/061_SPE_COMPLEXES").resolve()
template_file = Path("./templates/spe_ams.run").resolve()

# Set to True only if the HDF5 positions are fractional.
# From your printed arrays, they are likely already Cartesian.
POSITIONS_ARE_FRACTIONAL = False

# Write CIF files too?
WRITE_CIF = True

if not h5_root.exists():
    raise FileNotFoundError(f"HDF5 root folder not found: {h5_root}")

if not template_file.exists():
    raise FileNotFoundError(f"Template file not found: {template_file}")

out_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# -----------------------------
# Helpers
# -----------------------------

chemical_symbols = [
    "X",
    "H", "He",
    "Li", "Be", "B", "C", "N", "O", "F", "Ne",
    "Na", "Mg", "Al", "Si", "P", "S", "Cl", "Ar",
    "K", "Ca", "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Ga", "Ge", "As", "Se", "Br", "Kr",
    "Rb", "Sr", "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd",
    "In", "Sn", "Sb", "Te", "I", "Xe",
    "Cs", "Ba", "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd", "Tb", "Dy",
    "Ho", "Er", "Tm", "Yb", "Lu", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt",
    "Au", "Hg", "Tl", "Pb", "Bi", "Po", "At", "Rn"
]


def atomic_numbers_to_symbols(atomic_numbers):
    return [chemical_symbols[int(z)] for z in atomic_numbers]


def decode_if_bytes(x):
    if isinstance(x, bytes):
        return x.decode("utf-8")
    return str(x)


def clean_name(name):
    """
    Make a safe filename.
    """
    name = str(name)
    name = name.replace("/", "-")
    name = name.replace("\\", "-")
    name = re.sub(r"\s+", "_", name)
    return name


def parse_h5_filename(h5_file):
    """
    Example:
    MOF-545_AC_srd_seed1.h5 -> host=MOF-545, guest=AC, seed=srd_seed1
    NU-902_NAP_srd_seed2.h5 -> host=NU-902, guest=NAP, seed=srd_seed2
    """
    stem = h5_file.stem
    parts = stem.split("_")

    if len(parts) < 4:
        raise ValueError(f"Unexpected HDF5 filename format: {h5_file.name}")

    host = parts[0]
    guest = parts[1]
    seed = "_".join(parts[2:])

    return host, guest, seed

In [ ]:
# -----------------------------
# CIF helpers
# -----------------------------

def lattice_lengths_angles(lattice):
    """
    Convert 3 lattice vectors into CIF lengths and angles.
    lattice shape: (3, 3), rows are a, b, c vectors.
    """
    lattice = np.array(lattice, dtype=float)

    a_vec = lattice[0]
    b_vec = lattice[1]
    c_vec = lattice[2]

    a = np.linalg.norm(a_vec)
    b = np.linalg.norm(b_vec)
    c = np.linalg.norm(c_vec)

    alpha = np.degrees(np.arccos(np.dot(b_vec, c_vec) / (b * c)))
    beta = np.degrees(np.arccos(np.dot(a_vec, c_vec) / (a * c)))
    gamma = np.degrees(np.arccos(np.dot(a_vec, b_vec) / (a * b)))

    return a, b, c, alpha, beta, gamma


def cart_to_frac_manual(coords, lattice):
    """
    Convert Cartesian coordinates to fractional coordinates.
    Assumes lattice vectors are stored as rows: [a, b, c].
    Cartesian = fractional @ lattice
    Therefore fractional = Cartesian @ inverse(lattice)
    """
    coords = np.array(coords, dtype=float)
    lattice = np.array(lattice, dtype=float)

    return coords @ np.linalg.inv(lattice)


def write_cif_from_complex(output_file, symbols, coords, lattice, wrap_fractional=True):
    """
    Write a periodic CIF using fractional coordinates.

    symbols: list of element symbols
    coords: Cartesian coordinates, shape (n_atoms, 3)
    lattice: lattice vectors, shape (3, 3), rows are a, b, c
    """
    output_file = Path(output_file)

    coords = np.array(coords, dtype=float)
    lattice = np.array(lattice, dtype=float)

    a, b, c, alpha, beta, gamma = lattice_lengths_angles(lattice)

    frac_coords = cart_to_frac_manual(coords, lattice)

    if wrap_fractional:
        frac_coords = frac_coords % 1.0

    lines = []

    lines.append(f"data_{output_file.stem}")
    lines.append("_symmetry_space_group_name_H-M 'P1'")
    lines.append("_symmetry_Int_Tables_number 1")
    lines.append(f"_cell_length_a {a:.8f}")
    lines.append(f"_cell_length_b {b:.8f}")
    lines.append(f"_cell_length_c {c:.8f}")
    lines.append(f"_cell_angle_alpha {alpha:.8f}")
    lines.append(f"_cell_angle_beta {beta:.8f}")
    lines.append(f"_cell_angle_gamma {gamma:.8f}")
    lines.append("")
    lines.append("loop_")
    lines.append("_symmetry_equiv_pos_as_xyz")
    lines.append("'x, y, z'")
    lines.append("")
    lines.append("loop_")
    lines.append("_atom_site_label")
    lines.append("_atom_site_type_symbol")
    lines.append("_atom_site_fract_x")
    lines.append("_atom_site_fract_y")
    lines.append("_atom_site_fract_z")

    for i, (symbol, frac) in enumerate(zip(symbols, frac_coords), start=1):
        label = f"{symbol}{i}"
        lines.append(
            f"{label} {symbol} {frac[0]:.8f} {frac[1]:.8f} {frac[2]:.8f}"
        )

    output_file.write_text("\n".join(lines) + "\n")

In [ ]:
# -----------------------------
# Main loop: HDF5 -> AMS .run + CIF
# -----------------------------

h5_files = sorted(h5_root.glob("*/*.h5"))

print(f"Found {len(h5_files)} HDF5 files")

for h5_file in h5_files:
    try:
        host, guest, seed = parse_h5_filename(h5_file)
    except Exception as e:
        print(f"[skip] {h5_file.name}: {e}")
        continue

    output_dir = out_root / host / guest
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[H5] {h5_file.name}")
    print(f"Host: {host}")
    print(f"Guest: {guest}")
    print(f"Seed: {seed}")
    print(f"Output: {output_dir}")

    with h5py.File(h5_file, "r") as f:
        # Usually there is one top-level group, e.g. NU-902_acetaminophen
        group_name = list(f.keys())[0]
        group = f[group_name]

        complex_ids = [decode_if_bytes(x) for x in group["complex_ids"][:]]

        host_Z = group["host/atomic_numbers"][:]
        guest_Z = group["guest/atomic_numbers"][:]

        host_symbols = atomic_numbers_to_symbols(host_Z)
        guest_symbols = atomic_numbers_to_symbols(guest_Z)

        host_positions = group["host/positions"][:]           # shape: (n_host, 3)
        guest_positions_all = group["guest/positions"][:]     # shape: (n_complex, 1, n_guest, 3)

        lattice = group["lattice_vectors"][:]                 # shape: (3, 3)

        for i, complex_id in enumerate(complex_ids):
            complex_id_clean = clean_name(complex_id)

            # Guest coordinates for this complex
            guest_positions = guest_positions_all[i, 0]       # shape: (n_guest, 3)

            # Combine host + guest
            symbols = host_symbols + guest_symbols
            coords = np.vstack([host_positions, guest_positions])

            # Convert to Cartesian only if the HDF5 positions are fractional
            if POSITIONS_ARE_FRACTIONAL:
                coords = frac_to_cart(coords, lattice)

            name = f"{host}_{guest}_{seed}_{complex_id_clean}"

            output_run = output_dir / f"{name}.run"
            output_cif = output_dir / f"{name}.cif"

            # Generate AMS input file
            gen_inp_file(
                template_file,
                symbols=symbols,
                coords=coords.tolist(),
                lattice=lattice.tolist(),
                name=name,
                output=output_run,
            )

            print(f"  [OK] {output_run.name}")

            # Generate CIF file
            if WRITE_CIF:
                write_cif_from_complex(
                    output_file=output_cif,
                    symbols=symbols,
                    coords=coords,
                    lattice=lattice,
                    wrap_fractional=True,
                )

                print(f"  [OK] {output_cif.name}")

**MD pool.** For each retained MD frame, the guest and the single host unit cell
containing its centroid are extracted from the 2×2×2 supercell trajectory (with
minimum-image reconstruction if the guest straddles a periodic boundary), and an
AMS single-point input is written into `061_SPE_COMPLEXES/`. Frame extraction uses
[`scripts/helpers/lammps_guest_frame_extractor.py`](./helpers/lammps_guest_frame_extractor.py).

In [ ]:
from pathlib import Path
import sys

MODULE_DIR = Path("./helpers/").resolve()

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from lammps_guest_frame_extractor import (
    extract_frame_data,
    write_cif_from_data,
    list_frames,
)

from mof_toolkit import gen_inp_file

In [ ]:
ROOT = Path("../thesis/052_MD/0522_MD_trajectories").resolve()

TEMPLATE = Path("./templates/spe_ams.run").resolve()

NREPLICA = (2, 2, 2)

# For now, only extract the first frame.
FRAME_INDICES = list(range(0, 4000, 10))

# Later you can change this to, for example:
# FRAME_INDICES = [321]

In [ ]:
def find_host_data_file(host_dir: Path) -> Path | None:
    """
    Find the host-only LAMMPS data file inside a host folder.

    Expected example:
        ../0522_MD_trajectories/NU-902/data.NU-902_cond_dftb_geo-opt_EQeq
    """
    candidates = []

    for p in host_dir.iterdir():
        if not p.is_file():
            continue
        if not p.name.startswith("data"):
            continue
        if p.name == "data.initial_before_min":
            continue
        if "initial_before_min" in p.name:
            continue

        candidates.append(p)

    if not candidates:
        return None

    # Prefer the clean host EQeq file if present.
    eqeq = [p for p in candidates if "EQeq" in p.name]
    if len(eqeq) == 1:
        return eqeq[0]

    return sorted(candidates)[0]


def find_nve_trajectory(temp_dir: Path) -> Path | None:
    """
    Find the NVE trajectory inside a temperature/seed folder.

    Expected example:
        NU-902_AC_nve.lammpstrj
    """
    candidates = sorted(temp_dir.glob("*_nve.lammpstrj"))

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        print(f"More than one NVE trajectory found in {temp_dir}. Using:")
        print(candidates[0])
        return candidates[0]

    return None

In [ ]:
from pathlib import Path
import re

OUT_ROOT = Path("../thesis/061_SPE_COMPLEXES").resolve()


def parse_temp_seed(temp_seed: str) -> tuple[str, str]:
    """
    Convert folder name like:
        300K_seed_11914

    into:
        temperature = 300K
        seed = 11914
    """
    match = re.match(r"(?P<temp>\d+K)_seed_(?P<seed>\d+)", temp_seed)

    if match is None:
        raise ValueError(
            f"Could not parse temperature/seed folder name: {temp_seed}. "
            "Expected format like '300K_seed_11914'."
        )

    return match.group("temp"), match.group("seed")

In [ ]:
jobs = []
missing = []

for host_dir in sorted(ROOT.iterdir()):
    if not host_dir.is_dir():
        continue

    host_name = host_dir.name
    data_host = find_host_data_file(host_dir)

    if data_host is None:
        missing.append({
            "host": host_name,
            "guest": None,
            "temp_seed": None,
            "missing": "host data file",
            "path": str(host_dir),
        })
        continue

    for guest_dir in sorted(host_dir.iterdir()):
        if not guest_dir.is_dir():
            continue

        guest_name = guest_dir.name

        for temp_dir in sorted(guest_dir.iterdir()):
            if not temp_dir.is_dir():
                continue

            # Only process folders like 300K_seed_11914.
            if "K_seed_" not in temp_dir.name:
                continue

            data_system = temp_dir / "data.initial_before_min"
            trajectory = find_nve_trajectory(temp_dir)

            problems = []

            if not data_system.exists():
                problems.append("data.initial_before_min")

            if trajectory is None:
                problems.append("*_nve.lammpstrj")

            if problems:
                missing.append({
                    "host": host_name,
                    "guest": guest_name,
                    "temp_seed": temp_dir.name,
                    "missing": ", ".join(problems),
                    "path": str(temp_dir),
                })
                continue

            jobs.append({
                "host": host_name,
                "guest": guest_name,
                "temp_seed": temp_dir.name,
                "host_dir": host_dir,
                "guest_dir": guest_dir,
                "temp_dir": temp_dir,
                "data_host": data_host,
                "data_system": data_system,
                "trajectory": trajectory,
            })

print(f"Valid jobs found: {len(jobs)}")
print(f"Folders with missing files: {len(missing)}")

jobs[:3], missing[:3]

In [ ]:
for job in jobs:
    host = job["host"]
    guest = job["guest"]   # already AC, NAP, OXY, etc.

    temp_seed = job["temp_seed"]
    temperature, seed = parse_temp_seed(temp_seed)

    out_dir = OUT_ROOT / host / guest
    out_dir.mkdir(parents=True, exist_ok=True)

    for frame_index in FRAME_INDICES:
        data = extract_frame_data(
            trajectory=job["trajectory"],
            data_host=job["data_host"],
            data_system=job["data_system"],
            frame_index=frame_index,
            nreplica=NREPLICA,
            include_host=True,
        )

        name = f"{host}_{guest}_md_{temperature}_{seed}_F{frame_index}"

        cif_out = out_dir / f"{name}.cif"
        run_out = out_dir / f"{name}.run"

        write_cif_from_data(
            data,
            cif_out,
            name=name,
        )

        gen_inp_file(
            str(TEMPLATE),
            symbols=data["symbols"],
            coords=data["coords"],
            lattice=data["lattice"],
            name=name,
            output=str(run_out),
        )

        print(
            f"Written: {name}\n"
            f"  CIF: {cif_out}\n"
            f"  RUN: {run_out}\n"
            f"  host unit atoms = {data['metadata']['atoms_per_unit_cell']} | "
            f"guest atoms = {data['metadata']['guest_atoms_in_trajectory']} | "
            f"tile = {data['metadata']['tile']}"
        )

---
## 6. Minimum binding energy calculation

### 6.1 SPE screening of the configuration pool

A GFN1-xTB single-point energy is computed for every configuration (on the HPC).
The cells below scan each `.out` file for its total energy and compile per-pair
CSVs of the full pool.

> **⚠ Requires withheld data.** This section reads the bulky SPE `.out` files,
> which are not committed. Run it against your own
> `thesis/061_SPE_COMPLEXES/<host>/<guest>/*.out` outputs.

In [ ]:
from pathlib import Path
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm

In [ ]:
ROOT = Path("../thesis/061_SPE_COMPLEXES").resolve()

ROOT.exists(), ROOT

In [ ]:
ENERGY_KEY = b"Total Energy (hartree)"

MD_RE = re.compile(
    r"^(?P<host>.+?)_(?P<guest>[A-Za-z0-9]+)_md_"
    r"(?P<temperature>\d+K)_(?P<seed_id>\d+)_F(?P<frame>\d+)\.out$"
)

SRD_RE = re.compile(
    r"^(?P<host>.+?)_(?P<guest>[A-Za-z0-9]+)_srd_"
    r"(?P<seed>seed\d+)_complex_(?P<complex_id>\d+)\.out$"
)

FLOAT_RE = re.compile(
    rb"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[EeDd][-+]?\d+)?"
)


def parse_filename(path: Path) -> dict | None:
    """
    Parse:

        NU-902_AC_md_1200K_65200_F0.out
        NU-902_AC_srd_seed1_complex_17.out
    """
    name = path.name

    m = MD_RE.match(name)
    if m:
        d = m.groupdict()
        return {
            "host": d["host"],
            "guest_abbr": d["guest"],
            "method": "md",
            "seed": d["temperature"],       # 1200K
            "seed_id": d["seed_id"],        # 65200
            "id": f"F{d['frame']}",         # F0
        }

    m = SRD_RE.match(name)
    if m:
        d = m.groupdict()
        return {
            "host": d["host"],
            "guest_abbr": d["guest"],
            "method": "srd",
            "seed": d["seed"],              # seed1
            "seed_id": "",
            "id": f"C{d['complex_id']}",    # C17
        }

    return None


def extract_total_energy_hartree(path: Path) -> float | None:
    """
    Fast binary line scan.
    Stops as soon as 'Total Energy (hartree)' is found.
    """
    try:
        with path.open("rb") as f:
            for line in f:
                if ENERGY_KEY in line:
                    line = line.replace(b"D", b"E")
                    values = FLOAT_RE.findall(line)
                    if not values:
                        return None
                    return float(values[-1].decode("ascii"))
    except OSError:
        return None

    return None


def process_out_file(path: Path) -> dict:
    meta = parse_filename(path)

    if meta is None:
        return {
            "host": "",
            "guest_abbr": "",
            "method": "",
            "seed": "",
            "seed_id": "",
            "id": "",
            "total_energy_hartree": "",
            "out_file": str(path),
            "status": "filename_not_parsed",
        }

    energy = extract_total_energy_hartree(path)

    return {
        **meta,
        "total_energy_hartree": energy if energy is not None else "",
        "out_file": str(path),
        "status": "ok" if energy is not None else "energy_not_found",
    }

In [ ]:
def compile_host_guest_energies(
    guest_dir: Path,
    n_workers: int = 16,
    overwrite: bool = True,
) -> Path | None:
    """
    Compile all .out energies inside one host/guest folder.

    Example input:
        ../thesis/061_SPE_COMPLEXES/NU-902/AC/

    Output:
        ../thesis/061_SPE_COMPLEXES/NU-902/AC/NU-902_AC_total_energies.csv
    """
    host = guest_dir.parent.name
    guest = guest_dir.name

    csv_out = guest_dir / f"{host}_{guest}_total_energies.csv"

    if csv_out.exists() and not overwrite:
        print(f"Skipping existing CSV: {csv_out}")
        return csv_out

    out_files = sorted(guest_dir.glob("*.out"))

    if not out_files:
        print(f"No .out files found in: {guest_dir}")
        return None

    rows = []

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = [executor.submit(process_out_file, p) for p in out_files]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"{host}/{guest}",
            leave=False,
        ):
            rows.append(future.result())

    fieldnames = [
        "host",
        "guest_abbr",
        "method",
        "seed",
        "seed_id",
        "id",
        "total_energy_hartree",
        "out_file",
        "status",
    ]

    rows = sorted(
        rows,
        key=lambda r: (
            r["host"],
            r["guest_abbr"],
            r["method"],
            r["seed"],
            str(r["seed_id"]),
            r["id"],
        ),
    )

    with csv_out.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    ok = sum(r["status"] == "ok" for r in rows)
    bad = len(rows) - ok

    print(f"{host}/{guest}: wrote {csv_out.name} | ok={ok} | problems={bad}")

    return csv_out

In [ ]:
csv_files = []

for host_dir in sorted(ROOT.iterdir()):
    if not host_dir.is_dir():
        continue

    host = host_dir.name

    for guest_dir in sorted(host_dir.iterdir()):
        if not guest_dir.is_dir():
            continue

        csv_path = compile_host_guest_energies(
            guest_dir=guest_dir,
            n_workers=16,
            overwrite=True,
        )

        if csv_path is not None:
            csv_files.append(csv_path)

print(f"CSV files written: {len(csv_files)}")

In [ ]:
df = pd.read_csv(csv_files[0])
df.head()

In [ ]:
problem_rows = []

for csv_path in csv_files:
    df_tmp = pd.read_csv(csv_path)
    bad = df_tmp[df_tmp["status"] != "ok"]

    if len(bad) > 0:
        problem_rows.append(bad)

if problem_rows:
    problems = pd.concat(problem_rows, ignore_index=True)
    display(problems.head(50))
    print(f"Total problematic rows: {len(problems)}")
else:
    print("No problems found.")

In [ ]:
# Make one global summary from all per host-gust csv

all_dfs = [pd.read_csv(p) for p in csv_files]

df_all = pd.concat(all_dfs, ignore_index=True)

GLOBAL_CSV = ROOT / "all_total_energies.csv"
df_all.to_csv(GLOBAL_CSV, index=False)

print(f"Wrote global CSV: {GLOBAL_CSV}")
print(f"Total rows: {len(df_all)}")

### Candidate selection

Configurations are ranked by total complex energy. For each host–guest pair, only
poses within **+10 kcal/mol of the lowest-energy pose** are retained (a relative
steric-clash window), and the **top 20** lowest-energy structures are taken
forward to geometry optimization.

> **⚠ Reconcile with methodology.** Methodology §6.1 specifies selecting the top
> 20 *per sampling method* (SRD and MD independently, up to 40 candidates). The
> cell below currently selects the top 20 from the *combined* pool per pair — to
> match the documented per-method scheme, add `"method"` to `group_cols`. Confirm
> which scheme produced the committed results before finalizing.

In [ ]:
from pathlib import Path
import shutil
import re

import pandas as pd
from tqdm.auto import tqdm

In [ ]:
SPE_ROOT = Path("../thesis/061_SPE_COMPLEXES").resolve()
GEO_ROOT = Path("../thesis/062_GEO-OPT_COMPLEXES").resolve()

TOP_CSV = SPE_ROOT / "top20_within_10kcal_all_hosts_guests.csv"

HARTREE_TO_KCAL_MOL = 627.509474

ENERGY_WINDOW_KCAL = 10.0
TOP_N = 20

SPE_ROOT.exists(), GEO_ROOT

In [ ]:
csv_files = sorted(SPE_ROOT.rglob("*_total_energies.csv"))

# Avoid accidentally reading global summary files if they exist.
csv_files = [
    p for p in csv_files
    if p.name not in {
        "all_total_energies.csv",
        "all_total_energies_reduced.csv",
        "top20_within_10kcal_all_hosts_guests.csv",
    }
]

print(f"Found {len(csv_files)} per-host/guest CSV files")
csv_files[:5]

In [ ]:
dfs = []

for csv_path in csv_files:
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)

    # Keep only rows where energy extraction worked.
    if "status" in df.columns:
        df = df[df["status"] == "ok"].copy()

    df["total_energy_hartree"] = pd.to_numeric(
        df["total_energy_hartree"],
        errors="coerce",
    )

    df = df.dropna(subset=["total_energy_hartree"]).copy()

    # Store where this row came from.
    df["csv_file"] = str(csv_path)

    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

print(f"Total usable rows: {len(df_all)}")
df_all.head()

In [ ]:
selected_groups = []

group_cols = ["host", "guest_abbr"]

for (host, guest), g in df_all.groupby(group_cols, sort=True):
    g = g.copy()

    e_min = g["total_energy_hartree"].min()

    g["relative_energy_hartree"] = g["total_energy_hartree"] - e_min
    g["relative_energy_kcal_mol"] = g["relative_energy_hartree"] * HARTREE_TO_KCAL_MOL

    # Keep only structures within +10 kcal/mol of the minimum.
    g_window = g[g["relative_energy_kcal_mol"] <= ENERGY_WINDOW_KCAL].copy()

    # Then take the top 20 lowest-energy structures.
    g_top = (
        g_window
        .sort_values("total_energy_hartree", ascending=True)
        .head(TOP_N)
        .copy()
    )

    g_top["rank_within_host_guest"] = range(1, len(g_top) + 1)

    selected_groups.append(g_top)

df_top = pd.concat(selected_groups, ignore_index=True)

df_top = df_top.sort_values(
    ["host", "guest_abbr", "rank_within_host_guest"],
    ascending=True,
).reset_index(drop=True)

df_top.to_csv(TOP_CSV, index=False)

print(f"Wrote: {TOP_CSV}")
print(f"Selected rows: {len(df_top)}")
df_top.head(30)

### 6.2 Geometry-optimization inputs

Each selected candidate is copied into
`062_GEO-OPT_COMPLEXES/<host>/<guest>/<id>/` as an AMS geometry-optimization input
(GFN1-xTB/D3(BJ); atomic positions relaxed, lattice fixed), ready for HPC
submission.

In [ ]:
def c_id_to_number(c_id: str) -> str:
    """
    Convert C17 -> 17.
    """
    if not str(c_id).startswith("C"):
        raise ValueError(f"Expected SRD id like C17, got: {c_id}")
    return str(c_id)[1:]


def make_source_run_path(row) -> Path:
    """
    Source files live in:
        ../thesis/061_SPE_COMPLEXES/HOST/GUEST/

    MD example:
        NU-902_AC_md_1200K_65200_F0.run

    SRD example:
        NU-902_AC_srd_seed1_complex_17.run
    """
    host = row["host"]
    guest = row["guest_abbr"]
    method = row["method"]
    seed = row["seed"]
    seed_id = row["seed_id"]
    item_id = row["id"]

    base_dir = SPE_ROOT / host / guest

    if method == "md":
        filename = f"{host}_{guest}_md_{seed}_{seed_id}_{item_id}.run"

    elif method == "srd":
        complex_number = c_id_to_number(item_id)
        filename = f"{host}_{guest}_srd_{seed}_complex_{complex_number}.run"

    else:
        raise ValueError(f"Unknown method: {method}")

    return base_dir / filename


def make_destination_folder(row) -> Path:
    """
    Destination folders:

    MD:
        ../thesis/062_GEO-OPT_COMPLEXES/NU-902/AC/300K_11914_F4/

    SRD:
        ../thesis/062_GEO-OPT_COMPLEXES/NU-902/AC/seed1_C4/
    """
    host = row["host"]
    guest = row["guest_abbr"]
    method = row["method"]
    seed = row["seed"]
    seed_id = row["seed_id"]
    item_id = row["id"]

    if method == "md":
        folder_name = f"{seed}_{seed_id}_{item_id}"

    elif method == "srd":
        folder_name = f"{seed}_{item_id}"

    else:
        raise ValueError(f"Unknown method: {method}")

    return GEO_ROOT / host / guest / folder_name


def make_destination_run_name(row) -> str:
    """
    Destination run filenames:

    MD:
        NU-902_AC_300K_11914_F4.run

    SRD:
        NU-902_AC_seed1_complex_4.run
    """
    host = row["host"]
    guest = row["guest_abbr"]
    method = row["method"]
    seed = row["seed"]
    seed_id = row["seed_id"]
    item_id = row["id"]

    if method == "md":
        return f"{host}_{guest}_{seed}_{seed_id}_{item_id}.run"

    elif method == "srd":
        complex_number = c_id_to_number(item_id)
        return f"{host}_{guest}_{seed}_complex_{complex_number}.run"

    else:
        raise ValueError(f"Unknown method: {method}")


def convert_singlepoint_to_geoopt(text: str) -> str:
    """
    Replace SinglePoint with GeometryOptimization.
    This handles common AMS template variants.
    """
    text = text.replace("Task SinglePoint", "Task GeometryOptimization")
    text = text.replace("Task    SinglePoint", "Task GeometryOptimization")
    text = re.sub(
        r"\bSinglePoint\b",
        "GeometryOptimization",
        text,
        count=1,
    )
    return text

In [ ]:
missing = []

for _, row in df_top.iterrows():
    src = make_source_run_path(row)

    if not src.exists():
        missing.append({
            "host": row["host"],
            "guest_abbr": row["guest_abbr"],
            "method": row["method"],
            "seed": row["seed"],
            "seed_id": row["seed_id"],
            "id": row["id"],
            "expected_source": str(src),
        })

print(f"Missing source .run files: {len(missing)}")

if missing:
    df_missing = pd.DataFrame(missing)
    display(df_missing.head(50))

In [ ]:
created = []
failed = []

for _, row in tqdm(df_top.iterrows(), total=len(df_top)):
    src = make_source_run_path(row)
    dst_dir = make_destination_folder(row)
    dst_name = make_destination_run_name(row)
    dst = dst_dir / dst_name

    if not src.exists():
        failed.append({
            "reason": "source_run_missing",
            "source": str(src),
            "destination": str(dst),
        })
        continue

    dst_dir.mkdir(parents=True, exist_ok=True)

    try:
        text = src.read_text(encoding="utf-8", errors="replace")
        text = convert_singlepoint_to_geoopt(text)
        dst.write_text(text, encoding="utf-8")

        created.append({
            "host": row["host"],
            "guest_abbr": row["guest_abbr"],
            "method": row["method"],
            "seed": row["seed"],
            "seed_id": row["seed_id"],
            "id": row["id"],
            "rank_within_host_guest": row["rank_within_host_guest"],
            "relative_energy_kcal_mol": row["relative_energy_kcal_mol"],
            "source_run": str(src),
            "geoopt_run": str(dst),
        })

    except Exception as e:
        failed.append({
            "reason": repr(e),
            "source": str(src),
            "destination": str(dst),
        })

df_created = pd.DataFrame(created)
df_failed = pd.DataFrame(failed)

CREATED_CSV = GEO_ROOT / "created_geoopt_runs_from_top20.csv"
FAILED_CSV = GEO_ROOT / "failed_geoopt_runs_from_top20.csv"

GEO_ROOT.mkdir(parents=True, exist_ok=True)

df_created.to_csv(CREATED_CSV, index=False)

if len(df_failed) > 0:
    df_failed.to_csv(FAILED_CSV, index=False)

print(f"Created geo-opt run files: {len(df_created)}")
print(f"Failed: {len(df_failed)}")
print(f"Wrote log: {CREATED_CSV}")

if len(df_failed) > 0:
    print(f"Wrote failures: {FAILED_CSV}")
    display(df_failed.head(20))

The geometry-optimization inputs are ready. Run them on the HPC (one AMS job per
candidate), then continue below.

> **⚠ Requires withheld data.** The geo-opt `.out` files are not committed; the
> cells below parse your own outputs from `062_GEO-OPT_COMPLEXES/`.

### 6.3 Final binding energy

Once each selected candidate is optimized, its total energy and those of the
isolated host and guest (re-extracted from the relaxed complex using the known
host atom count) are read, and `BE = E_complex − (E_host + E_guest)` is evaluated
at the optimized geometry. The cells below define the parsers and run them over
`062_GEO-OPT_COMPLEXES/`.

In [ ]:
! pip install tqdm
from pathlib import Path
import subprocess
import shlex
import re
import math

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from mof_toolkit import gen_inp_file, frac_to_cart

In [ ]:
GEO_ROOT = Path("../thesis/062_GEO-OPT_COMPLEXES").resolve()

TEMPLATE = Path("./templates/spe_ams.run").resolve()

HOST_ATOMS_PER_UNIT_CELL = {
    "NU-902": 210,
    "PCN-223": 276,
    "PCN-225": 420,
    "MOF-545": 630,
}

if not GEO_ROOT.exists():
    raise FileNotFoundError(f"GEO_ROOT not found: {GEO_ROOT}")

if not TEMPLATE.exists():
    raise FileNotFoundError(f"TEMPLATE not found: {TEMPLATE}")

GEO_ROOT

In [ ]:
def clean_cif_number(value: str) -> float:
    """
    Convert CIF numbers like:
        12.345(6)
    into:
        12.345
    """
    value = str(value).strip().strip("'").strip('"')
    value = re.sub(r"\([^)]*\)$", "", value)
    return float(value)


def infer_element_from_label(label: str) -> str:
    """
    Infer element from atom labels such as:
        C1, O12, Zr3, H_
    """
    label = str(label).strip().strip("'").strip('"')
    m = re.match(r"([A-Z][a-z]?)", label)
    if m is None:
        raise ValueError(f"Could not infer element from CIF label: {label}")
    return m.group(1)


def lattice_from_cell(a, b, c, alpha_deg, beta_deg, gamma_deg):
    """
    Build a 3x3 lattice matrix from CIF cell parameters.

    Returns row vectors:
        [[ax, ay, az],
         [bx, by, bz],
         [cx, cy, cz]]
    """
    alpha = math.radians(alpha_deg)
    beta = math.radians(beta_deg)
    gamma = math.radians(gamma_deg)

    va = np.array([a, 0.0, 0.0])

    vb = np.array([
        b * math.cos(gamma),
        b * math.sin(gamma),
        0.0,
    ])

    cx = c * math.cos(beta)
    cy = c * (math.cos(alpha) - math.cos(beta) * math.cos(gamma)) / math.sin(gamma)
    cz_sq = c**2 - cx**2 - cy**2

    if cz_sq < 0 and abs(cz_sq) < 1e-8:
        cz_sq = 0.0

    vc = np.array([
        cx,
        cy,
        math.sqrt(cz_sq),
    ])

    return np.vstack([va, vb, vc]).tolist()

In [ ]:
def read_cif_symbols_frac_lattice(cif_path: Path):
    """
    Read symbols, fractional coordinates, and lattice from a CIF file.

    This expects the CIF to contain:
        _cell_length_a
        _cell_length_b
        _cell_length_c
        _cell_angle_alpha
        _cell_angle_beta
        _cell_angle_gamma

    and an atom_site loop with fractional coordinates:
        _atom_site_fract_x
        _atom_site_fract_y
        _atom_site_fract_z

    It uses _atom_site_type_symbol if present.
    Otherwise, it infers the element from _atom_site_label.
    """
    lines = cif_path.read_text(encoding="utf-8", errors="replace").splitlines()

    cell = {}

    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        parts = stripped.split()
        if len(parts) < 2:
            continue

        key = parts[0]
        value = parts[1]

        if key in {
            "_cell_length_a",
            "_cell_length_b",
            "_cell_length_c",
            "_cell_angle_alpha",
            "_cell_angle_beta",
            "_cell_angle_gamma",
        }:
            cell[key] = clean_cif_number(value)

    required_cell = [
        "_cell_length_a",
        "_cell_length_b",
        "_cell_length_c",
        "_cell_angle_alpha",
        "_cell_angle_beta",
        "_cell_angle_gamma",
    ]

    missing_cell = [k for k in required_cell if k not in cell]
    if missing_cell:
        raise ValueError(f"Missing CIF cell parameters in {cif_path}: {missing_cell}")

    lattice = lattice_from_cell(
        cell["_cell_length_a"],
        cell["_cell_length_b"],
        cell["_cell_length_c"],
        cell["_cell_angle_alpha"],
        cell["_cell_angle_beta"],
        cell["_cell_angle_gamma"],
    )

    symbols = []
    frac_coords = []

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if line != "loop_":
            i += 1
            continue

        i += 1
        headers = []

        while i < len(lines) and lines[i].strip().startswith("_"):
            headers.append(lines[i].strip())
            i += 1

        if not any(h.startswith("_atom_site") for h in headers):
            continue

        try:
            ix = headers.index("_atom_site_fract_x")
            iy = headers.index("_atom_site_fract_y")
            iz = headers.index("_atom_site_fract_z")
        except ValueError:
            continue

        if "_atom_site_type_symbol" in headers:
            isym = headers.index("_atom_site_type_symbol")
            ilabel = None
        elif "_atom_site_label" in headers:
            isym = None
            ilabel = headers.index("_atom_site_label")
        else:
            raise ValueError(f"No atom symbol or label field found in {cif_path}")

        while i < len(lines):
            raw = lines[i].strip()

            if (
                not raw
                or raw.startswith("#")
            ):
                i += 1
                continue

            if raw == "loop_" or raw.startswith("_") or raw.startswith("data_"):
                break

            parts = shlex.split(raw)

            if len(parts) < len(headers):
                i += 1
                continue

            if isym is not None:
                symbol = parts[isym].strip()
            else:
                symbol = infer_element_from_label(parts[ilabel])

            x = clean_cif_number(parts[ix])
            y = clean_cif_number(parts[iy])
            z = clean_cif_number(parts[iz])

            symbols.append(symbol)
            frac_coords.append([x, y, z])

            i += 1

        if symbols:
            break

    if not symbols:
        raise ValueError(f"No atom_site fractional coordinates found in {cif_path}")

    return symbols, frac_coords, lattice

In [ ]:
def find_geoopt_out_file(complex_dir: Path) -> Path | None:
    """
    Find the AMS .out file inside one complex folder.

    Excludes common scheduler output files like slurm*.out.
    """
    out_files = sorted(
        p for p in complex_dir.glob("*.out")
        if not p.name.startswith("slurm")
    )

    if len(out_files) == 0:
        return None

    if len(out_files) == 1:
        return out_files[0]

    # Prefer files that look like the actual complex output.
    # If there are several, choose the largest one, which is usually the AMS output.
    return max(out_files, key=lambda p: p.stat().st_size)


def run_ams_get_structure(out_file: Path, cif_file: Path, overwrite: bool = False):
    """
    Run:
        ams_get_structure -input file.out -output file.cif
    """
    if cif_file.exists() and not overwrite:
        return "exists"

    cmd = [
        "ams_get_structure",
        "-input", str(out_file),
        "-output", str(cif_file),
    ]

    result = subprocess.run(
        cmd,
        cwd=str(out_file.parent),
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"ams_get_structure failed for {out_file}\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    return "created"


def split_host_guest_from_cif(cif_file: Path, host: str):
    """
    Read optimized complex CIF and split into:
        host symbols/coords/lattice
        guest symbols/coords

    Host is periodic, guest is non-periodic.
    Both coordinates are Cartesian.
    """
    if host not in HOST_ATOMS_PER_UNIT_CELL:
        raise KeyError(
            f"Host {host} not in HOST_ATOMS_PER_UNIT_CELL. "
            f"Available: {list(HOST_ATOMS_PER_UNIT_CELL)}"
        )

    n_host = HOST_ATOMS_PER_UNIT_CELL[host]

    symbols, frac_coords, lattice = read_cif_symbols_frac_lattice(cif_file)

    if len(symbols) <= n_host:
        raise ValueError(
            f"CIF has {len(symbols)} atoms, but host {host} needs {n_host}. "
            f"No guest atoms remain."
        )

    cart_coords = frac_to_cart(frac_coords, lattice)

    host_symbols = symbols[:n_host]
    host_coords = cart_coords[:n_host]

    guest_symbols = symbols[n_host:]
    guest_coords = cart_coords[n_host:]

    return {
        "symbols": symbols,
        "frac_coords": frac_coords,
        "cart_coords": cart_coords,
        "lattice": lattice,
        "host": {
            "symbols": host_symbols,
            "coords": host_coords,
            "n_atoms": len(host_symbols),
        },
        "guest": {
            "symbols": guest_symbols,
            "coords": guest_coords,
            "n_atoms": len(guest_symbols),
        },
    }

In [ ]:
def process_complex_folder(
    complex_dir: Path,
    template: Path,
    overwrite_cif: bool = False,
    overwrite_run: bool = True,
):
    """
    For one folder like:
        ../062_GEO-OPT_COMPLEXES/NU-902/AC/seed1_C4/

    Create:
        *_opt.cif
        NU-902.run
        AC.run
    """
    guest_dir = complex_dir.parent
    host_dir = guest_dir.parent

    guest = guest_dir.name
    host = host_dir.name

    out_file = find_geoopt_out_file(complex_dir)

    if out_file is None:
        return {
            "status": "missing_out",
            "host": host,
            "guest": guest,
            "complex_folder": complex_dir.name,
            "complex_dir": str(complex_dir),
            "out_file": "",
            "cif_file": "",
            "host_run": "",
            "guest_run": "",
            "host_atoms": "",
            "guest_atoms": "",
        }

    cif_file = complex_dir / f"{out_file.stem}_opt.cif"

    try:
        cif_status = run_ams_get_structure(
            out_file=out_file,
            cif_file=cif_file,
            overwrite=overwrite_cif,
        )

        structure = split_host_guest_from_cif(
            cif_file=cif_file,
            host=host,
        )

        host_run = complex_dir / f"{host}.run"
        guest_run = complex_dir / f"{guest}.run"

        if overwrite_run or not host_run.exists():
            gen_inp_file(
                str(template),
                symbols=structure["host"]["symbols"],
                coords=structure["host"]["coords"],
                lattice=structure["lattice"],
                fractional=False,
                name=host,
                output=str(host_run),
            )

        if overwrite_run or not guest_run.exists():
            gen_inp_file(
                str(template),
                symbols=structure["guest"]["symbols"],
                coords=structure["guest"]["coords"],
                name=guest,
                output=str(guest_run),
            )

        return {
            "status": "ok",
            "host": host,
            "guest": guest,
            "complex_folder": complex_dir.name,
            "complex_dir": str(complex_dir),
            "out_file": str(out_file),
            "cif_file": str(cif_file),
            "cif_status": cif_status,
            "host_run": str(host_run),
            "guest_run": str(guest_run),
            "host_atoms": structure["host"]["n_atoms"],
            "guest_atoms": structure["guest"]["n_atoms"],
        }

    except Exception as e:
        return {
            "status": "failed",
            "reason": repr(e),
            "host": host,
            "guest": guest,
            "complex_folder": complex_dir.name,
            "complex_dir": str(complex_dir),
            "out_file": str(out_file),
            "cif_file": str(cif_file),
            "host_run": "",
            "guest_run": "",
            "host_atoms": "",
            "guest_atoms": "",
        }

In [ ]:
complex_dirs = []

for host_dir in sorted(GEO_ROOT.iterdir()):
    if not host_dir.is_dir():
        continue

    host = host_dir.name

    if host not in HOST_ATOMS_PER_UNIT_CELL:
        print(f"Skipping unknown host folder: {host}")
        continue

    for guest_dir in sorted(host_dir.iterdir()):
        if not guest_dir.is_dir():
            continue

        for complex_dir in sorted(guest_dir.iterdir()):
            if not complex_dir.is_dir():
                continue

            complex_dirs.append(complex_dir)

print(f"Complex folders found: {len(complex_dirs)}")
complex_dirs[:5]

In [ ]:
logs = []

for complex_dir in tqdm(complex_dirs):
    log = process_complex_folder(
        complex_dir=complex_dir,
        template=TEMPLATE,
        overwrite_cif=False,
        overwrite_run=True,
    )
    logs.append(log)

df_log = pd.DataFrame(logs)

LOG_CSV = GEO_ROOT / "extract_host_guest_runs_from_geoopt_outputs_log.csv"
df_log.to_csv(LOG_CSV, index=False)

print(f"Wrote log: {LOG_CSV}")
df_log["status"].value_counts()

In [ ]:
df_log[df_log["status"] != "ok"].head(50)

### Reported minimum binding energy

The reported BE for each host–guest pair is the minimum over all relaxed
candidates. These per-pair minima are saved as the primary results and carried
into [`selectivity-analysis`](./selectivity-analysis.ipynb).

**Next:** [`selectivity-analysis.ipynb`](./selectivity-analysis.ipynb) — topology
ranking, descriptor correlations, and CBD-referenced selectivity (§7).

In [ ]:
from pathlib import Path
import re
import csv

import pandas as pd
from tqdm.auto import tqdm

In [ ]:
GEO_ROOT = Path("../thesis/062_GEO-OPT_COMPLEXES").resolve()

HARTREE_TO_KCAL_MOL = 627.509474

BE_CSV = GEO_ROOT / "binding_energies_geoopt_kcal.csv"
PROBLEM_CSV = GEO_ROOT / "binding_energies_geoopt_problems.csv"

if not GEO_ROOT.exists():
    raise FileNotFoundError(f"GEO_ROOT not found: {GEO_ROOT}")

GEO_ROOT

In [ ]:
ENERGY_KEY = b"Total Energy (hartree)"

FLOAT_RE = re.compile(
    rb"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[EeDd][-+]?\d+)?"
)


def extract_total_energy_hartree(out_file: Path) -> float | None:
    """
    Extract the Total Energy (hartree) from an AMS .out file.

    Reads line by line and stops as soon as the energy line is found.
    """
    try:
        with out_file.open("rb") as f:
            for line in f:
                if ENERGY_KEY in line:
                    line = line.replace(b"D", b"E")
                    values = FLOAT_RE.findall(line)

                    if not values:
                        return None

                    return float(values[-1].decode("ascii"))

    except OSError:
        return None

    return None

In [ ]:
MD_FOLDER_RE = re.compile(
    r"^(?P<temperature>\d+K)_(?P<seed_id>\d+)_(?P<frame>F\d+)$"
)

SRD_FOLDER_RE = re.compile(
    r"^(?P<seed>seed\d+)_(?P<complex_id>C\d+)$"
)


def parse_structure_folder(folder_name: str) -> dict | None:
    """
    Parse structure folders like:

        300K_11914_F530
        seed1_C4
    """
    m = MD_FOLDER_RE.match(folder_name)
    if m:
        d = m.groupdict()
        return {
            "method": "md",
            "seed": d["temperature"],
            "seed_id": d["seed_id"],
            "structure_id": d["frame"],
        }

    m = SRD_FOLDER_RE.match(folder_name)
    if m:
        d = m.groupdict()
        return {
            "method": "srd",
            "seed": d["seed"],
            "seed_id": "",
            "structure_id": d["complex_id"],
        }

    return None


def c_id_to_number(c_id: str) -> str:
    """
    Convert C4 -> 4.
    """
    if not str(c_id).startswith("C"):
        raise ValueError(f"Expected complex id like C4, got: {c_id}")
    return str(c_id)[1:]

In [ ]:
def expected_complex_out_name(host: str, guest: str, meta: dict) -> str:
    """
    Expected complex .out filename.

    MD:
        NU-902_AC_300K_11914_F530.out

    SRD:
        NU-902_AC_seed1_complex_4.out
    """
    method = meta["method"]
    seed = meta["seed"]
    seed_id = meta["seed_id"]
    structure_id = meta["structure_id"]

    if method == "md":
        return f"{host}_{guest}_{seed}_{seed_id}_{structure_id}.out"

    if method == "srd":
        complex_number = c_id_to_number(structure_id)
        return f"{host}_{guest}_{seed}_complex_{complex_number}.out"

    raise ValueError(f"Unknown method: {method}")


def find_required_out_files(structure_dir: Path, host: str, guest: str, meta: dict):
    """
    Return the complex, host, and guest .out files inside one structure folder.

    Expected:
        complex: HOST_GUEST_....out
        host:    HOST.out
        guest:   GUEST.out
    """
    complex_expected = structure_dir / expected_complex_out_name(host, guest, meta)
    host_expected = structure_dir / f"{host}.out"
    guest_expected = structure_dir / f"{guest}.out"

    missing = []

    if not complex_expected.exists():
        missing.append(str(complex_expected))

    if not host_expected.exists():
        missing.append(str(host_expected))

    if not guest_expected.exists():
        missing.append(str(guest_expected))

    return complex_expected, host_expected, guest_expected, missing

In [ ]:
def process_structure_folder(structure_dir: Path) -> tuple[dict | None, dict | None]:
    """
    Process one folder:

        GEO_ROOT/HOST/GUEST/STRUCTURE_ID/

    Returns:
        row, problem

    Only one of them should be non-None.
    """
    guest_dir = structure_dir.parent
    host_dir = guest_dir.parent

    host = host_dir.name
    guest = guest_dir.name
    folder_name = structure_dir.name

    meta = parse_structure_folder(folder_name)

    if meta is None:
        return None, {
            "status": "structure_folder_not_parsed",
            "host": host,
            "guest": guest,
            "structure_folder": folder_name,
            "path": str(structure_dir),
            "details": "Expected folder like 300K_11914_F530 or seed1_C4",
        }

    complex_out, host_out, guest_out, missing = find_required_out_files(
        structure_dir=structure_dir,
        host=host,
        guest=guest,
        meta=meta,
    )

    if missing:
        return None, {
            "status": "missing_out_files",
            "host": host,
            "guest": guest,
            "method": meta["method"],
            "seed": meta["seed"],
            "seed_id": meta["seed_id"],
            "structure_id": meta["structure_id"],
            "structure_folder": folder_name,
            "path": str(structure_dir),
            "details": " | ".join(missing),
        }

    complex_E_hartree = extract_total_energy_hartree(complex_out)
    host_E_hartree = extract_total_energy_hartree(host_out)
    guest_E_hartree = extract_total_energy_hartree(guest_out)

    missing_energy = []

    if complex_E_hartree is None:
        missing_energy.append(str(complex_out))

    if host_E_hartree is None:
        missing_energy.append(str(host_out))

    if guest_E_hartree is None:
        missing_energy.append(str(guest_out))

    if missing_energy:
        return None, {
            "status": "energy_not_found",
            "host": host,
            "guest": guest,
            "method": meta["method"],
            "seed": meta["seed"],
            "seed_id": meta["seed_id"],
            "structure_id": meta["structure_id"],
            "structure_folder": folder_name,
            "path": str(structure_dir),
            "details": " | ".join(missing_energy),
        }

    BE_hartree = complex_E_hartree - (host_E_hartree + guest_E_hartree)

    row = {
        "host": host,
        "guest": guest,
        "method": meta["method"],
        "seed": meta["seed"],
        "seed_id": meta["seed_id"],
        "structure_id": meta["structure_id"],

        # Energies converted to kcal/mol.
        # These absolute values will be very large, but BE is the important one.
        "complex_E_kcal_mol": complex_E_hartree * HARTREE_TO_KCAL_MOL,
        "host_E_kcal_mol": host_E_hartree * HARTREE_TO_KCAL_MOL,
        "guest_E_kcal_mol": guest_E_hartree * HARTREE_TO_KCAL_MOL,
        "BE_kcal_mol": BE_hartree * HARTREE_TO_KCAL_MOL,

        # Also keep hartree values for checking/reproducibility.
        "complex_E_hartree": complex_E_hartree,
        "host_E_hartree": host_E_hartree,
        "guest_E_hartree": guest_E_hartree,
        "BE_hartree": BE_hartree,

        "complex_out": str(complex_out),
        "host_out": str(host_out),
        "guest_out": str(guest_out),
        "structure_folder": str(structure_dir),
    }

    return row, None

In [ ]:
structure_dirs = []

for host_dir in sorted(GEO_ROOT.iterdir()):
    if not host_dir.is_dir():
        continue

    for guest_dir in sorted(host_dir.iterdir()):
        if not guest_dir.is_dir():
            continue

        for structure_dir in sorted(guest_dir.iterdir()):
            if not structure_dir.is_dir():
                continue

            structure_dirs.append(structure_dir)

print(f"Structure folders found: {len(structure_dirs)}")
structure_dirs[:5]

In [ ]:
rows = []
problems = []

for structure_dir in tqdm(structure_dirs):
    row, problem = process_structure_folder(structure_dir)

    if row is not None:
        rows.append(row)

    if problem is not None:
        problems.append(problem)

df_be = pd.DataFrame(rows)
df_problems = pd.DataFrame(problems)

print(f"Successful BE rows: {len(df_be)}")
print(f"Problems: {len(df_problems)}")

In [ ]:
if len(df_be) > 0:
    df_be = df_be.sort_values(
        ["host", "guest", "method", "seed", "seed_id", "structure_id"],
        ascending=True,
    ).reset_index(drop=True)

    df_be.to_csv(BE_CSV, index=False)

    print(f"Wrote binding energy CSV:")
    print(BE_CSV)

if len(df_problems) > 0:
    df_problems.to_csv(PROBLEM_CSV, index=False)

    print(f"Wrote problem log:")
    print(PROBLEM_CSV)

### 6.4 Sampling and convergence diagnostics

Three summary tables quantify how trustworthy each binding-energy minimum is. They
are computed from the **raw per-configuration / per-frame energy pools** — the SRD
single-point pool and the MD trajectories — which are too large to commit; the
committed CSVs in `070_CORRELATION/` are their published outputs, and the cell below
regenerates them when the raw pools are present. The statistical definitions:

- **`csq_sampling_summary.csv`** — sampling-volume bias in the large-pore csq host.
  For each guest, over its $N$ csq single-point configurations, $d_i$ is the closest
  guest-atom–framework contact in configuration $i$; the table reports
  $\overline{d}$, the median, and the wall-contact fractions
  $\%_{\le 7\,\text{Å}} = 100\,\lvert\{i: d_i \le 7\}\rvert / N$ and $\%_{\le 5\,\text{Å}}$.
  A low fraction means the guest rarely sampled the pore wall, so its minimum is a
  sampling floor rather than a true optimum.
- **`md_convergence_summary.csv`** — for each (topology, guest, temperature $T$) MD
  binding-energy series $\{B_t\}_{t=1}^{n}$, the first-half/second-half drift
  $\mathrm{diff} = \frac{2}{n}\sum_{t\le n/2} B_t - \frac{2}{n}\sum_{t> n/2} B_t$
  (kcal/mol). $\lvert\mathrm{diff}\rvert$ small relative to the BE scale indicates an
  equilibrated trajectory.
- **`be_outliers.csv`** — raw minima far beyond the physical bulk range
  ($\lvert BE\rvert$ typically 34–97 kcal/mol). These arise from GFN1-xTB fragment
  misassignment (spurious proton transfer) at compressed MD-sourced geometries and
  are *excluded* from `minimum_binding_energies.csv`; the `likely_mechanism` column
  is annotated by inspection of the offending geometry.

In [ ]:
# 6.4 — regenerate the sampling/convergence diagnostics from the raw pools.
import numpy as np, pandas as pd
from pathlib import Path

CORR    = Path("../thesis/070_CORRELATION")
SRD_RAW = Path("../thesis/051_SRD")                       # single-point config pool (uncommitted)
MD_RAW  = Path("../thesis/052_MD/0522_MD_trajectories")   # MD trajectories (uncommitted)


def csq_sampling_summary(srd_root):
    '''Wall-contact statistics for every guest in the csq (MOF-545) config pool.'''
    rows = []
    for system in sorted(srd_root.glob("MOF-545_*")):
        d = closest_framework_contact_per_config(system)     # Å, one value per config
        rows.append(dict(system=system.name, n_used=len(d),
                         mean_dist=d.mean(), median_dist=float(np.median(d)),
                         pct_within_7A=100 * np.mean(d <= 7), pct_within_5A=100 * np.mean(d <= 5)))
    return pd.DataFrame(rows)


def md_convergence_summary(md_root):
    '''First-half vs second-half BE drift for every (topo, guest, T) trajectory.'''
    rows = []
    for series in md_root.rglob("*_be_series.npy"):
        B = np.load(series); h = len(B) // 2
        topo, guest, T = parse_md_trajectory_name(series)
        rows.append(dict(topo=topo, guest=guest, T=T, n_frames=len(B),
                         diff=B[:h].mean() - B[h:].mean()))
    return pd.DataFrame(rows)


def be_outliers(raw_minima, bulk_lo=-97.0, bulk_hi=-34.0):
    '''Flag raw minima well beyond the physical bulk range as fragment-misassignment.'''
    flagged = raw_minima[raw_minima.minBE_kcal_mol < bulk_lo * 1.3].copy()
    flagged["typical_range_min"] = bulk_lo
    flagged["typical_range_max"] = bulk_hi
    flagged["magnitude_ratio_vs_typical"] = (flagged.minBE_kcal_mol.abs() / 70.7).round(1)
    return flagged


if SRD_RAW.exists():
    csq_sampling_summary(SRD_RAW).to_csv(CORR / "csq_sampling_summary.csv", index=False)
    print("regenerated csq_sampling_summary.csv")
else:
    print("csq_sampling_summary.csv: raw SRD pool absent — using committed summary",
          f"({len(pd.read_csv(CORR / 'csq_sampling_summary.csv'))} guests)")

if MD_RAW.exists():
    md_convergence_summary(MD_RAW).to_csv(CORR / "md_convergence_summary.csv", index=False)
    print("regenerated md_convergence_summary.csv")
else:
    print("md_convergence_summary.csv: raw MD trajectories absent — using committed summary",
          f"({len(pd.read_csv(CORR / 'md_convergence_summary.csv'))} rows)")

print("be_outliers.csv: committed curated list —",
      f"{len(pd.read_csv(CORR / 'be_outliers.csv'))} flagged pairs (mechanism annotated by inspection)")